In [4]:
# ---- Reproducibility
import random
import torch
import sys
import os
import matplotlib.pyplot as plt
from cebmf_torch import cEBMF
#seed = 1234
#random.seed(seed)
#torch.manual_seed(seed)
#if torch.cuda.is_available():
#    torch.cuda.manual_seed_all(seed)


In [5]:
from cebmf_torch.cebmf.cebmf import normal_means_loglik

In [6]:


# ---- Parameters
n, p = 50, 40
noise_std = 0.1

# (Optional) choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---- Data generation (PyTorch)
# Use float64 to mirror NumPy defaults
u = torch.rand(n, dtype=torch.float64, device=device)          # length n
v = torch.rand(p, dtype=torch.float64, device=device)          # length p

# Rank-1 matrix via outer product
rank_1_matrix = torch.outer(u, v)                              # (n, p)

# Homoscedastic Gaussian noise
noise = noise_std * torch.randn(n, p, dtype=torch.float64, device=device)

noisy_matrix = rank_1_matrix + noise

# ---- Print (move to CPU for readability if needed)
print("Rank-1 Matrix (Outer Product):")
print(rank_1_matrix.cpu().numpy())

print("\nNoisy Matrix (with Homoscedastic Noise):")
print(noisy_matrix.cpu().numpy())


Rank-1 Matrix (Outer Product):
[[0.28187708 0.02278465 0.40464521 ... 0.05282415 0.28553462 0.64850005]
 [0.3407602  0.02754428 0.48917416 ... 0.06385892 0.34518179 0.78396942]
 [0.20688666 0.01672303 0.29699361 ... 0.03877084 0.20957114 0.47597345]
 ...
 [0.10933606 0.00883784 0.15695605 ... 0.02048973 0.11075477 0.25154384]
 [0.05833319 0.00471518 0.08373949 ... 0.01093172 0.0590901  0.13420415]
 [0.25721064 0.02079081 0.3692356  ... 0.04820162 0.26054811 0.59175124]]

Noisy Matrix (with Homoscedastic Noise):
[[ 0.26198453  0.09105375  0.41938725 ...  0.00840613  0.25690636
   0.60023728]
 [ 0.26661704 -0.17049694  0.44782761 ...  0.05072773  0.50087754
   0.69864586]
 [ 0.11174734 -0.13527029  0.33113305 ... -0.08587616  0.25386895
   0.45120828]
 ...
 [ 0.0863731   0.16678529  0.19521068 ...  0.09278858  0.05898622
   0.20459018]
 [ 0.03061228 -0.02373965  0.18156192 ... -0.04689639  0.06449547
   0.1067752 ]
 [ 0.43334889 -0.05044793  0.4693104  ...  0.12010664  0.33817308
   0.71

In [8]:
mycebmf=  cEBMF(data= noisy_matrix) 
mycebmf.initialise_factors()
print(mycebmf.L[:,1])
print(mycebmf.F[:,1])

tensor([ 0.0209, -0.5135, -0.4321,  0.2992, -0.0317, -0.1288, -0.0787, -0.1819,
         0.1213, -0.2348,  0.4045,  0.1181, -0.0612, -0.0177, -0.2739, -0.0127,
        -0.0709,  0.3336,  0.2006,  0.0388, -0.0974, -0.0415, -0.0109, -0.1102,
         0.3553,  0.1574, -0.0862, -0.0083,  0.4238, -0.0283,  0.2852, -0.0866,
        -0.0568, -0.0805,  0.2183,  0.0548,  0.1546, -0.0135, -0.1682,  0.2944,
         0.0793, -0.0180, -0.0661, -0.0108, -0.1318,  0.0177,  0.0668,  0.1719,
        -0.2591, -0.0218])
tensor([ 0.2140,  0.2253,  0.0598, -0.0097, -0.1638,  0.0214, -0.1835,  0.1008,
        -0.0223,  0.0100,  0.3194,  0.3899,  0.2065,  0.1036, -0.1916, -0.0916,
        -0.1453, -0.1087, -0.1011, -0.0192,  0.0123, -0.0839, -0.1786,  0.1026,
        -0.1213, -0.1858, -0.1201,  0.0631,  0.1953, -0.2879,  0.1710,  0.0796,
         0.0556, -0.0440,  0.1543, -0.0311, -0.2176,  0.1517, -0.2393,  0.0269])


In [9]:
tau_map=None
eps = 1e-12

In [12]:
k = 0
eps = 1e-12
tau_map = None if mycebmf.noise.type == "constant" else mycebmf.tau_map

# ---------- Update L[:, k] ----------
Rk = mycebmf._partial_residual_masked(k)  # (N,P), zeros where missing
fk = mycebmf.F[:, k]  # (P,)
fk2 = mycebmf.F2[:, k]  # (P,)

if tau_map is None:
    denom_l = (fk2.view(1, -1) * mycebmf.mask).sum(dim=1).clamp_min(eps)  # (N,)
    num_l = Rk @ fk  # (N,)
    se_l = torch.sqrt(1.0 / (mycebmf.tau * denom_l))
else:
    denom_l = (tau_map * (fk2.view(1, -1) * mycebmf.mask)).sum(dim=1).clamp_min(eps)
    num_l = (tau_map * Rk) @ fk
    se_l = torch.sqrt(1.0 / denom_l)

lhat = num_l / denom_l

X_model = mycebmf._build_covariate_matrix(
    external_cov=mycebmf.covariate.X_l,
    self_cov_enabled=mycebmf.covariate.self_row_cov,
    factors=mycebmf.L,
    k=k,
    dim_size=mycebmf.N,
)
with torch.enable_grad():
    resL = mycebmf.prior_L_fn.fit(
        X=X_model,
        betahat=lhat,
        sebetahat=se_l,
        internal_epoch=mycebmf.internal_epoch,
        model_param=mycebmf.model_state_L[k],
    )
with torch.no_grad():
    mycebmf.model_state_L[k] = resL.model_param
    mycebmf.L[:, k] = resL.post_mean
    mycebmf.L2[:, k] = resL.post_mean2
    nm_ll_L = normal_means_loglik(x=lhat, s=se_l, Et=resL.post_mean, Et2=resL.post_mean2)
    mycebmf.kl_l[k] = torch.as_tensor((-resL.loss) - nm_ll_L, device=mycebmf.device)
    mycebmf.pi0_L[k] = resL.pi0_null if hasattr(resL, "pi0_null") else None

# ---------- Update F[:, k] ----------
Rk = mycebmf._partial_residual_masked(k)  # recompute with updated L
lk = mycebmf.L[:, k]  # (N,)
lk2 = mycebmf.L2[:, k]  # (N,)

if tau_map is None:
    denom_f = (lk2.view(-1, 1) * mycebmf.mask).sum(dim=0).clamp_min(eps)  # (P,)
    num_f = Rk.T @ lk  # (P,)
    se_f = torch.sqrt(1.0 / (mycebmf.tau * denom_f))
else:
    denom_f = (tau_map * (lk2.view(-1, 1) * mycebmf.mask)).sum(dim=0).clamp_min(eps)
    num_f = (tau_map * Rk).T @ lk
    se_f = torch.sqrt(1.0 / denom_f)

fhat = num_f / denom_f

X_model = mycebmf._build_covariate_matrix(
    external_cov=mycebmf.covariate.X_f,
    self_cov_enabled=mycebmf.covariate.self_col_cov,
    factors=mycebmf.F,
    k=k,
    dim_size=mycebmf.P,
)
with torch.enable_grad():
    resF = mycebmf.prior_F_fn.fit(
        X=X_model,
        betahat=fhat,
        sebetahat=se_f,
        internal_epoch=mycebmf.internal_epoch,
        model_param=mycebmf.model_state_F[k],
    )
with torch.no_grad():
    mycebmf.model_state_F[k] = resF.model_param
    mycebmf.F[:, k] = resF.post_mean
    mycebmf.F2[:, k] = resF.post_mean2
    nm_ll_F = normal_means_loglik(x=fhat, s=se_f, Et=resF.post_mean, Et2=resF.post_mean2)
    mycebmf.kl_f[k] = torch.as_tensor((-resF.loss) - nm_ll_F, device=mycebmf.device)
    mycebmf.pi0_F[k] = resF.pi0_null if hasattr(resF, "pi0_null") else None

In [ ]:
mycebmf=  cEBMF(data= noisy_matrix) 
mycebmf.initialise_factors()


In [ ]:
mycebmf.iter_once()

In [ ]:
mycebmf0=  cEBMF(data= noisy_matrix) 
mycebmf0.initialise_factors()
print(mycebmf0.L[:,1])
print(mycebmf0.F[:,1])

In [ ]:
R = mycebmf0.Y0 - mycebmf0.L @ mycebmf0.F.T

In [ ]:
k= 0 


In [ ]:
# Precondition: R is the global residual with ALL components removed:
# R = Y - sum_j L[:, j] @ F[:, j]^T

for k in range(mycebmf0.K):
    # 1) Add back k's current contribution to form the partial residual
    R_k = R + torch.outer(mycebmf0.L[:, k], mycebmf0.F[:, k])  # (N,P)

    # ---------- Update L[:, k] (hold F[:, k] fixed) ----------
    fk  = mycebmf0.F[:, k]      # (P,)
    fk2 = mycebmf0.F2[:, k]     # (P,)

    if tau_map is None:
        denom_l = ((fk2.view(1, -1) * mycebmf0.mask).sum(dim=1).clamp_min(eps))  # (N,)
        num_l   = R_k @ fk                                                        # (N,)
        se_l    = torch.sqrt(1.0 / (mycebmf0.tau * denom_l))
    else:
        denom_l = ((tau_map * (fk2.view(1, -1) * mycebmf0.mask)).sum(dim=1).clamp_min(eps))  # (N,)
        num_l   = (tau_map * R_k) @ fk                                                        # (N,)
        se_l    = torch.sqrt(1.0 / denom_l)

    lhat = num_l / denom_l

    X_model = mycebmf0.update_cov_L(k)
    with torch.enable_grad():
        resL = mycebmf0.prior_L_fn(
            X=X_model, betahat=lhat, sebetahat=se_l, model_param=mycebmf0.model_state_L[k],
        )
    with torch.no_grad():
        mycebmf0.model_state_L[k] = resL.model_param
        mycebmf0.L[:, k]  = resL.post_mean
        mycebmf0.L2[:, k] = resL.post_mean2
        nm_ll_L = normal_means_loglik(x=lhat, s=se_l, Et=resL.post_mean, Et2=resL.post_mean2)
        mycebmf0.kl_l[k] = torch.as_tensor((-resL.loss) - nm_ll_L, device=mycebmf0.device)
        mycebmf0.pi0_L[k] = resL.pi0_null if hasattr(resL, "pi0_null") else None

    # 2) Recompute global residual with UPDATED L[:, k] (F[:, k] still old)
    R = R_k - torch.outer(mycebmf0.L[:, k], mycebmf0.F[:, k])

    # 3) Add back k's (now with updated L) to update F
    R_k = R + torch.outer(mycebmf0.L[:, k], mycebmf0.F[:, k])

    # ---------- Update F[:, k] (hold UPDATED L[:, k] fixed) ----------
    lk  = mycebmf0.L[:, k]      # (N,)
    lk2 = mycebmf0.L2[:, k]     # (N,)

    if tau_map is None:
        denom_f = ((lk2.view(-1, 1) * mycebmf0.mask).sum(dim=0).clamp_min(eps))  # (P,)
        num_f   = R_k.T @ lk                                                     # (P,)
        se_f    = torch.sqrt(1.0 / (mycebmf0.tau * denom_f))
    else:
        denom_f = ((tau_map * (lk2.view(-1, 1) * mycebmf0.mask)).sum(dim=0).clamp_min(eps))  # (P,)
        num_f   = (tau_map * R_k).T @ lk                                                     # (P,)
        se_f    = torch.sqrt(1.0 / denom_f)

    fhat = num_f / denom_f

    X_model = mycebmf0.update_cov_F(k)
    with torch.enable_grad():
        resF = mycebmf0.prior_F_fn(
            X=X_model, betahat=fhat, sebetahat=se_f, model_param=mycebmf0.model_state_F[k],
        )
    with torch.no_grad():
        mycebmf0.model_state_F[k] = resF.model_param
        mycebmf0.F[:, k]  = resF.post_mean
        mycebmf0.F2[:, k] = resF.post_mean2
        nm_ll_F = normal_means_loglik(x=fhat, s=se_f, Et=resF.post_mean, Et2=resF.post_mean2)
        mycebmf0.kl_f[k] = torch.as_tensor((-resF.loss) - nm_ll_F, device=mycebmf0.device)
        mycebmf0.pi0_F[k] = resF.pi0_null if hasattr(resF, "pi0_null") else None

    # 4) Recompute global residual with UPDATED F[:, k]
    R = R_k - torch.outer(mycebmf0.L[:, k], mycebmf0.F[:, k])


In [ ]:
plt.scatter(mycebmf0.F[:, 1],mycebmf.F[:, 1 ])

In [ ]:
# R is the global residual with ALL components removed:
# R = Y - sum_j L[:, j] @ F[:, j]^T

for k in range(mycebmf0.K):
    Lk  = mycebmf0.L[:, k]
    Fk  = mycebmf0.F[:, k]
    Lk2 = mycebmf0.L2[:, k]
    Fk2 = mycebmf0.F2[:, k]

    # ---- Add back k's contribution: R <- R + Lk Fk^T (in-place, no outer allocation)
    R.addr_(Lk, Fk, alpha=1.0)

    # ---------- Update L[:, k] ----------
    if tau_map is None:
        # denom_l = sum_j mask[i,j] * Fk2[j]
        denom_l = mycebmf0.mask @ Fk2
        # num_l = (R @ Fk)
        num_l   = R @ Fk
        se_l    = torch.sqrt(1.0 / (mycebmf0.tau * denom_l.clamp_min(eps)))
    else:
        # denom_l = sum_j tau_map[i,j]*mask[i,j]*Fk2[j]
        denom_l = (tau_map * mycebmf0.mask) @ Fk2
        # num_l = sum_j tau_map[i,j]*R[i,j]*Fk[j]
        num_l   = torch.einsum('ij,ij,j->i', R, tau_map, Fk)
        se_l    = torch.sqrt(1.0 / denom_l.clamp_min(eps))

    lhat = num_l / denom_l.clamp_min(eps)

    X_model = mycebmf0.update_cov_L(k)
    with torch.enable_grad():
        resL = mycebmf0.prior_L_fn(
            X=X_model, betahat=lhat, sebetahat=se_l, model_param=mycebmf0.model_state_L[k]
        )
    with torch.no_grad():
        mycebmf0.model_state_L[k] = resL.model_param
        mycebmf0.L[:, k]  = resL.post_mean
        mycebmf0.L2[:, k] = resL.post_mean2
        nm_ll_L = normal_means_loglik(x=lhat, s=se_l, Et=resL.post_mean, Et2=resL.post_mean2)
        mycebmf0.kl_l[k]  = torch.as_tensor((-resL.loss) - nm_ll_L, device=mycebmf0.device)
        mycebmf0.pi0_L[k] = resL.pi0_null if hasattr(resL, "pi0_null") else None

    # ---- Subtract UPDATED Lk contribution: R <- R - Lk_new Fk^T
    Lk = mycebmf0.L[:, k]  # updated
    R.addr_(Lk, Fk, alpha=-1.0)

    # ---- Add back (updated Lk) to prepare F update: R <- R + Lk_new Fk^T
    R.addr_(Lk, Fk, alpha=1.0)

    # ---------- Update F[:, k] ----------
    if tau_map is None:
        # denom_f = sum_i mask[i,j] * Lk2[i]
        denom_f = mycebmf0.mask.T @ mycebmf0.L2[:, k]
        # num_f = (R^T @ Lk)
        num_f   = R.T @ Lk
        se_f    = torch.sqrt(1.0 / (mycebmf0.tau * denom_f.clamp_min(eps)))
    else:
        # denom_f = sum_i tau_map[i,j]*mask[i,j]*Lk2[i]
        denom_f = (tau_map * mycebmf0.mask).transpose(0, 1) @ mycebmf0.L2[:, k]
        # num_f = sum_i tau_map[i,j]*R[i,j]*Lk[i]
        num_f   = torch.einsum('ij,ij,i->j', R, tau_map, Lk)
        se_f    = torch.sqrt(1.0 / denom_f.clamp_min(eps))

    fhat = num_f / denom_f.clamp_min(eps)

    X_model = mycebmf0.update_cov_F(k)
    with torch.enable_grad():
        resF = mycebmf0.prior_F_fn(
            X=X_model, betahat=fhat, sebetahat=se_f, model_param=mycebmf0.model_state_F[k]
        )
    with torch.no_grad():
        mycebmf0.model_state_F[k] = resF.model_param
        mycebmf0.F[:, k]  = resF.post_mean
        mycebmf0.F2[:, k] = resF.post_mean2
        nm_ll_F = normal_means_loglik(x=fhat, s=se_f, Et=resF.post_mean, Et2=resF.post_mean2)
        mycebmf0.kl_f[k]  = torch.as_tensor((-resF.loss) - nm_ll_F, device=mycebmf0.device)
        mycebmf0.pi0_F[k] = resF.pi0_null if hasattr(resF, "pi0_null") else None

    # ---- Subtract UPDATED Fk contribution: R <- R - Lk_new Fk_new^T
    Fk = mycebmf0.F[:, k]  # updated
    R.addr_(Lk, Fk, alpha=-1.0)


In [ ]:
plt.scatter(mycebmf0.F[:, 0],mycebmf.F[:, 0 ])